# Phase 2 — primitive discovery

Fit the same whole-trial split to the observed trajectory `q` and endpoint residual `f`. FADA uses the local Fourier anechoic-demixing implementation; SCA calls the official Python package. This notebook does not select a winning representation or primitive model automatically.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_primitives.paths import PROJECT_ROOT
from motion_primitives.preprocessing import load_collections
from motion_primitives.fada import FADA, prepare_spectra

# ---- experiment settings ----
COLLECTION = PROJECT_ROOT / 'collections/3D-ARM-Gaze/custom-phase200-v1'
VIEW = 'successful'          # use 'all' to include failed trials
JOINT_NAMES = None
SEED = 42
COMPONENTS = [2, 3, 4, 5, 6]

RUN_FADA = True
FADA_MODEL = 'spatiotemporal'   # temporal | spatiotemporal | space_by_time
FADA_BACKEND = 'torch'
FADA_DEVICE = 'auto'
FADA_BATCH_SIZE = 512
FADA_WORKERS = 8
FADA_PAD = 100
FADA_ENERGY = 0.995
FADA_ITERATIONS = 20
FADA_DELAY_STEPS = 40
FADA_MAX_DELAY = 20
FADA_RESTARTS = 2

RUN_SCA = True
SCA_EPOCHS = 3000

OUTPUT = PROJECT_ROOT / 'notebooks/results/03_Primitive_Discovery' / COLLECTION.name / VIEW
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
collection = load_collections([COLLECTION], JOINT_NAMES, view=VIEW)
Q, F = collection['q'], collection['f']
metadata = collection['metadata'].reset_index(drop=True)
joints = collection['joint_names']

split_path = OUTPUT / 'split.csv'
if split_path.exists():
    saved = pd.read_csv(split_path)
    if saved.trial_id.tolist() != metadata.trial_id.tolist():
        raise ValueError('Saved split does not match this ordered trial cohort.')
    split = saved.split.to_numpy()
else:
    split = train_validation_test_split(metadata, seed=SEED)
    pd.DataFrame({'trial_id': metadata.trial_id, 'split': split}).to_csv(split_path, index=False)

display(pd.Series(split).value_counts().rename('trials').to_frame())
print(f'{len(metadata)} trials, {Q.shape[1]} phase samples, {Q.shape[2]} joints')


In [ ]:
provenance = {
    'collection': str(COLLECTION), 'view': VIEW, 'seed': SEED,
    'components': COMPONENTS, 'joint_names': joints,
    'representations': ['q', 'f'],
    'fada': {'model': FADA_MODEL, 'backend': FADA_BACKEND, 'device': FADA_DEVICE,
             'batch_size': FADA_BATCH_SIZE, 'max_delay': FADA_MAX_DELAY,
             'restarts': FADA_RESTARTS},
    'sca': {'epochs': SCA_EPOCHS},
}
(OUTPUT / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')


In [ ]:
def nmse(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    denominator = np.sum((actual - actual.mean(axis=0, keepdims=True)) ** 2)
    return float(np.sum((actual - predicted) ** 2) / denominator)


def fit_sca(values, split, *, n_components=4, n_epochs=3000, seed=42):
    from sca.models import SCA

    discovery = np.asarray(split) == 'discovery'
    np.random.seed(seed)
    model = SCA(n_components=n_components, n_epochs=n_epochs)
    model.fit(values[discovery].reshape(-1, values.shape[-1]))

    flat = values.reshape(-1, values.shape[-1])
    latent = model.transform(flat).reshape(len(values), values.shape[1], n_components)
    reconstruction = model.reconstruct(flat).reshape(values.shape)
    return model, latent, reconstruction


## FADA

Each fit learns the primitive library only from `discovery` trials. Validation and test trials are projected onto that fixed library. The identical split is used for `q` and `f`.


In [ ]:
fada_rows = []
if RUN_FADA:
    discovery = split == 'discovery'
    for representation, values in [('q', Q), ('f', F)]:
        spectra, mean, cumulative_energy = prepare_spectra(
            values, discovery, pad=FADA_PAD, energy=FADA_ENERGY)
        n_time = values.shape[1] + 2 * FADA_PAD
        fourier_k = spectra.shape[-1] - 1

        device = FADA_DEVICE
        if FADA_BACKEND == 'torch' and device == 'auto':
            import torch
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        elif FADA_BACKEND != 'torch':
            device = 'cpu'

        for k in COMPONENTS:
            print(f'FADA {representation=} {k=} {FADA_BACKEND=} {device=}')
            fada = FADA(
                n_time=n_time,
                k=fourier_k,
                max_delay=FADA_MAX_DELAY,
                iterations=FADA_ITERATIONS,
                delay_steps=FADA_DELAY_STEPS,
                workers=FADA_WORKERS,
                backend=FADA_BACKEND,
                device=device,
                batch_size=FADA_BATCH_SIZE,
            )

            if FADA_MODEL == 'temporal':
                train = spectra[discovery].reshape(-1, 1, fourier_k + 1)
                fit = fada.fit(train, (k,), restarts=FADA_RESTARTS, seed=SEED)
                coefficients, delays, fitted_spectra = fada.project(
                    spectra.reshape(-1, 1, fourier_k + 1), fit['library'])
                fitted_spectra = fitted_spectra.reshape(spectra.shape)
                coefficients = coefficients.reshape(len(values), values.shape[2], -1)
                delays = delays.reshape(len(values), values.shape[2], -1)
            elif FADA_MODEL == 'spatiotemporal':
                fit = fada.fit(spectra[discovery], (k,), restarts=FADA_RESTARTS, seed=SEED)
                coefficients, delays, fitted_spectra = fada.project(spectra, fit['library'])
            elif FADA_MODEL == 'space_by_time':
                fit = fada.fit(
                    spectra[discovery], (k, values.shape[2]),
                    restarts=FADA_RESTARTS, seed=SEED)
                coefficients, delays, fitted_spectra = fada.project(spectra, fit['library'])
            else:
                raise ValueError('Unknown FADA_MODEL')

            time_domain = np.fft.irfft(fitted_spectra, n=n_time, axis=-1)
            reconstruction = time_domain[..., FADA_PAD:FADA_PAD + values.shape[1]].transpose(0, 2, 1)
            reconstruction = reconstruction + mean[None, None, :]

            folder = OUTPUT / 'fada' / representation / f'k{k:02d}'
            folder.mkdir(parents=True, exist_ok=True)
            arrays = {'coefficients': coefficients, 'delays': delays,
                      'mean': mean, 'history': fit['history']}
            arrays.update({f'library_{name}': value for name, value in fit['library'].items()})
            np.savez_compressed(folder / 'fit.npz', **arrays)

            for split_name in ('discovery', 'validation', 'test'):
                mask = split == split_name
                score = nmse(values[mask], reconstruction[mask])
                fada_rows.append({
                    'representation': representation, 'k': k,
                    'model': FADA_MODEL, 'backend': FADA_BACKEND, 'device': device,
                    'split': split_name, 'trials': int(mask.sum()),
                    'nmse': score, 'variance_explained': 1.0 - score,
                    'fourier_k': fourier_k,
                })

fada_summary = pd.DataFrame(fada_rows)
if len(fada_summary):
    fada_summary.to_csv(OUTPUT / 'fada_summary.csv', index=False)
    display(fada_summary)


## Sparse Component Analysis

SCA is fit only on discovery samples after reshaping `(trial, phase, joint)` to `(sample, joint)`. Validation/test trajectories are transformed and reconstructed with the fitted model; no held-out sample is used for fitting.


In [ ]:
sca_rows = []
if RUN_SCA:
    for representation, values in [('q', Q), ('f', F)]:
        for k in COMPONENTS:
            print(f'SCA {representation=} {k=}')
            model, latent, reconstruction = fit_sca(
                values, split, n_components=k, n_epochs=SCA_EPOCHS, seed=SEED)
            folder = OUTPUT / 'sca' / representation / f'k{k:02d}'
            folder.mkdir(parents=True, exist_ok=True)
            trial_activity = np.mean(np.abs(latent), axis=1)
            arrays = {'trial_activity': trial_activity}
            if hasattr(model, 'params') and isinstance(model.params, dict):
                arrays.update({f'param_{name}': value for name, value in model.params.items()})
            np.savez_compressed(folder / 'fit.npz', **arrays)
            for split_name in ('discovery', 'validation', 'test'):
                mask = split == split_name
                score = nmse(values[mask], reconstruction[mask])
                sca_rows.append({'representation': representation, 'k': k,
                                 'split': split_name, 'trials': int(mask.sum()),
                                 'nmse': score, 'variance_explained': 1.0 - score})

sca_summary = pd.DataFrame(sca_rows)
if len(sca_summary):
    sca_summary.to_csv(OUTPUT / 'sca_summary.csv', index=False)
    display(sca_summary)


## Comparison

The first comparison is held-out reconstruction as a function of component count. Reuse, stability across repeated fits/bootstrap samples, and cross-method component matching are subsequent Phase-2 analyses; they should be added only after the basic fits are numerically validated.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for method, table in [('FADA', fada_summary), ('SCA', sca_summary)]:
    if not len(table):
        continue
    test = table[table.split == 'test']
    for representation in ('q', 'f'):
        rows = test[test.representation == representation].sort_values('k')
        ax.plot(rows.k, rows.nmse, marker='o', label=f'{method} {representation}')
ax.set(xlabel='Number of components', ylabel='Held-out NMSE', title='Phase-2 held-out reconstruction')
ax.legend(frameon=False)
fig.savefig(OUTPUT / 'heldout_nmse.png')
plt.show()
